In [107]:
!pip install hyperopt


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [108]:
import hyperopt
print(hyperopt.__version__)

0.3.0


In [109]:
from hyperopt import hp

search_space = {
    "x":hp.quniform("x",-10,10,1), # x -10부터 10까지 1씩증가
    "y":hp.quniform("y",-15,15,1)
}



In [110]:
from hyperopt import STATUS_OK

def objective_func(search_space): #목적함수 사용자정의함수로 만듦
    x = search_space["x"] #변수 x 지정한거 1씩 증가시키는걸로 위에서 만든거 넣음
    y = search_space["y"]
    retval = x**2 - 20*y 
    return retval

In [111]:
from hyperopt import fmin, tpe, Trials
import numpy as np

trial_val = Trials()

best_01 = fmin(
    fn = objective_func, 
    space = search_space, 
    algo = tpe.suggest, 
    max_evals = 5, 
    trials = trial_val, 
    rstate = np.random.default_rng(seed = 0)
    )
print(best_01)

100%|██████████| 5/5 [00:00<00:00, 1796.58trial/s, best loss: -224.0]
{'x': np.float64(-4.0), 'y': np.float64(12.0)}


In [112]:
from hyperopt import fmin, tpe, Trials
import numpy as np

trial_val = Trials()

best_01 = fmin(fn = objective_func,
               space = search_space,
               algo = tpe.suggest,
               max_evals = 20,
               trials = trial_val,
               rstate = np.random.default_rng(seed = 0))

print(best_01)

100%|██████████| 20/20 [00:00<00:00, 1628.92trial/s, best loss: -296.0]
{'x': np.float64(2.0), 'y': np.float64(15.0)}


In [113]:
import pandas as pd

losses = [loss_dict["loss"] for loss_dict in trial_val.results]

result_df = pd.DataFrame({
    "x": trial_val.vals["x"],
    "y": trial_val.vals["y"],
    "losses" : losses
})

result_df

,x,y,losses
0,-6.0,5.0,-64.0
1,-4.0,10.0,-184.0
2,4.0,-2.0,56.0
3,-4.0,12.0,-224.0
4,9.0,1.0,61.0
5,2.0,15.0,-296.0
6,10.0,7.0,-40.0
7,-9.0,-10.0,281.0
8,-8.0,0.0,64.0
9,-0.0,-5.0,100.0


In [114]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

dataset = load_breast_cancer()

df = pd.DataFrame(data = dataset.data, columns = dataset.feature_names)
df["target"] = dataset.target

X_features = df.iloc[:,:-1]
y_label = df.iloc[:,-1]


In [115]:
X_train, X_test, y_train, y_test = train_test_split(
    X_features,
    y_label,
    test_size = 0.2,
    random_state = 42
)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train,
    y_train,
    test_size = 0.1,
    random_state = 42
)

In [116]:
xgb_search_space = {
    "max_depth": hp.quniform("max_depth",5,20,1),
    "min_child_weight" : hp.quniform("min_child_weight",1,2,1),
    "learning_rate": hp.quniform("learning_rate",0.01,0.2,0.1),
    "colsample_bytree": hp.quniform("colsample_bytree",0.5,1,0.1)
}

In [117]:
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier
from hyperopt import STATUS_OK

def objective_func(search_space):
    xgb_clf = XGBClassifier(
        n_estimators = 100,
        max_depth = int(search_space["max_depth"]),
        min_child_weight = int(search_space["min_child_weight"]),
        learning_rate = search_space["learning_rate"],
        colsample_bytree = search_space["colsample_bytree"],
        eval_metric = "logloss"
        )
    accuracy = cross_val_score(
        xgb_clf, 
        X_train, 
        y_train, 
        scoring = "accuracy", 
        cv = 3
        )
    return {"loss": -1 * np.mean(accuracy), "status": STATUS_OK}

In [118]:
from hyperopt import fmin, tpe, Trials

trial_val = Trials()
best = fmin(
    fn = objective_func,
    space = xgb_search_space,
    algo = tpe.suggest,
    max_evals = 50,
    trials = trial_val,
    rstate = np.random.default_rng(seed = 0)
    )

print(best)

100%|██████████| 50/50 [00:06<00:00,  7.49trial/s, best loss: -0.9670326478447775]
{'colsample_bytree': np.float64(0.8), 'learning_rate': np.float64(0.2), 'max_depth': np.float64(6.0), 'min_child_weight': np.float64(1.0)}


In [119]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def get_clf_eval(y_test, pred=None, pred_proba=None):
    confusion = confusion_matrix(y_test, pred)
    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred)
    recall = recall_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    roc_auc = roc_auc_score(y_test, pred_proba)

    print(confusion)
    print(accuracy,precision,recall,f1,roc_auc)

    print("정확도: {0:.4f}, 정밀도:{1:.4f}, 재현율:{2:.4f}, F1:{3:.4f},AUC:{4:.4f}".format(accuracy, precision, recall, f1, roc_auc))

In [121]:
xgb_wrapper = XGBClassifier(
    n_estimators = 400,
    learning_rate = round(best["learning_rate"],5),
    max_depth = int(best["max_depth"]),
    min_child_weight = int(best["min_child_weight"]),
    colsample_bytree = round(best["colsample_bytree"],5),
    early_stopping_rounds = 50,
    eval_metric = "logloss", 
    )

evals = [(X_tr, y_tr), (X_val, y_val)]

xgb_wrapper.fit(
    X_tr, 
    y_tr, 
    eval_set = evals, 
    verbose = True
    )

pred = xgb_wrapper.predict(X_test)
pred_prob = xgb_wrapper.predict_proba(X_test)[:,1]

get_clf_eval(y_test, pred, pred_prob)

[0]	validation_0-logloss:0.50431	validation_1-logloss:0.50198
[1]	validation_0-logloss:0.40041	validation_1-logloss:0.40636
[2]	validation_0-logloss:0.32681	validation_1-logloss:0.33037
[3]	validation_0-logloss:0.27024	validation_1-logloss:0.28468
[4]	validation_0-logloss:0.22486	validation_1-logloss:0.24467
[5]	validation_0-logloss:0.18971	validation_1-logloss:0.21899
[6]	validation_0-logloss:0.16161	validation_1-logloss:0.19186
[7]	validation_0-logloss:0.13785	validation_1-logloss:0.16795
[8]	validation_0-logloss:0.11864	validation_1-logloss:0.15814
[9]	validation_0-logloss:0.10264	validation_1-logloss:0.14423
[10]	validation_0-logloss:0.08916	validation_1-logloss:0.13251
[11]	validation_0-logloss:0.07857	validation_1-logloss:0.12664
[12]	validation_0-logloss:0.07007	validation_1-logloss:0.12293
[13]	validation_0-logloss:0.06191	validation_1-logloss:0.11666
[14]	validation_0-logloss:0.05563	validation_1-logloss:0.11277
[15]	validation_0-logloss:0.05005	validation_1-logloss:0.11057
[1